In [44]:
import os
from pypdf import PdfReader

In [45]:
pdf_path = "C:/Ijjat me learning/RAG Project/documents/machine_learning.pdf"

reader=PdfReader(pdf_path)

print("Total Pages : ", len(reader.pages))

Total Pages :  6


In [46]:
all_text=" "

for page_no, page in enumerate(reader.pages):
    text= page.extract_text()

    if text:
        all_text += text + "\n"

    print(f"Page {page_no + 1} extracted")


Page 1 extracted
Page 2 extracted
Page 3 extracted
Page 4 extracted
Page 5 extracted
Page 6 extracted


In [47]:
print(all_text)

 Learning notes  Beginner to intermediate
Page 1
 Machine Learning
 A practical foundation: concepts, workflow, algorithms and evaluation
How to use this guide
Read one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete
the exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.
Learning outcomes

Explain supervised, unsupervised and reinforcement learning.

Distinguish classification from regression.

Prepare a dataset without leaking test information.

Train a baseline model and evaluate it with suitable metrics.

Recognise overfitting and explain ways to reduce it.
Prerequisites
Basic Python is helpful. You should be comfortable with variables, functions, lists and importing packages. Basic
algebra and averages are enough to begin; learn probability and statistics alongside the guide.

Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine

In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

In [49]:
chunks=text_splitter.split_text(all_text)

print("Total chunks: ", len(chunks))

Total chunks:  27


In [50]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk)


--- Chunk 1 ---

Learning notes  Beginner to intermediate
Page 1
 Machine Learning
 A practical foundation: concepts, workflow, algorithms and evaluation
How to use this guide
Read one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete
the exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.
Learning outcomes

Explain supervised, unsupervised and reinforcement learning.


--- Chunk 2 ---

Learning outcomes

Explain supervised, unsupervised and reinforcement learning.

Distinguish classification from regression.

Prepare a dataset without leaking test information.

Train a baseline model and evaluate it with suitable metrics.

Recognise overfitting and explain ways to reduce it.
Prerequisites
Basic Python is helpful. You should be comfortable with variables, functions, lists and importing packages. Basic

--- Chunk 3 ---

algebra and averages are enough to begin; lear

In [51]:
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("all-MiniLM-L6-v2")

chunk_embedding = model.encode(
    chunks,
    show_progress_bar=True
)

print("Number of chunks: ", len(chunks))
print("Embedding shape:", chunk_embedding.shape)

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]

Number of chunks:  27
Embedding shape: (27, 384)


In [52]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [chunk_embedding[0]],
    [chunk_embedding[1]]
)

print(similarity[0][0])

0.61570585


In [53]:
print(cosine_similarity(
    [chunk_embedding[0]],
    [chunk_embedding[2]]
)[0][0])

0.44167385


In [54]:
%pip install -U chromadb

Note: you may need to restart the kernel to use updated packages.


c:\Ijjat me learning\RAG Project\rag_project\Scripts\python.exe: No module named pip


In [80]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="rag_documents",
    metadata={"hnsw:space": "cosine"}
)

In [56]:
ids = [f"chunk_{i}" for i in range(len(chunks))]

metadatas = [
    {"source": "machine_learning.pdf"}
    for _ in chunks
]

collection.upsert(
    ids=ids,
    documents=chunks,
    embeddings=chunk_embedding.tolist(),
    metadatas=metadatas
)

print("Total records:", collection.count())

Total records: 27


In [105]:
collection_demo = chroma_client.get_or_create_collection(
    name="metadata_demo"
)

collection_demo.add(
    ids=["doc_1"],
    documents=["Machine learning is a branch of AI."],
    embeddings=[model.encode(
        "Machine learning is a branch of AI."
    ).tolist()],
    metadatas=[
        {
            "source": "machine_learning.pdf",
            "category": "Machine Learning"
        }
    ]
)

In [58]:
print("Total records:", collection.count())

Total records: 27


In [59]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

print(result)

{'ids': ['chunk_0', 'chunk_1', 'chunk_2'], 'embeddings': None, 'documents': ['Learning notes \x7f Beginner to intermediate\nPage 1\n Machine Learning\n A practical foundation: concepts, workflow, algorithms and evaluation\nHow to use this guide\nRead one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete\nthe exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.\nLearning outcomes\n\x7f\nExplain supervised, unsupervised and reinforcement learning.\n\x7f', 'Learning outcomes\n\x7f\nExplain supervised, unsupervised and reinforcement learning.\n\x7f\nDistinguish classification from regression.\n\x7f\nPrepare a dataset without leaking test information.\n\x7f\nTrain a baseline model and evaluate it with suitable metrics.\n\x7f\nRecognise overfitting and explain ways to reduce it.\nPrerequisites\nBasic Python is helpful. You should be comfortable with variables, functions, lists and

In [60]:
query = "What is machine learning?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [61]:
for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(document)


Result 1:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
learning algorithm.
Example: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or

Result 2:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam, churn/no churn, disease class.

Regression predicts a numeric value: house price, delivery time, sales.
Unsupervised learning

Result 3:
not spam. The model learns statistical patterns that help classify new messages.
Traditional programming v

In [62]:
questions = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is generative AI?",
    "How does deep learning work?",
    "What is supervised learning?"
]

for question in questions:

    print("\nQUESTION:", question)

    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    for i, document in enumerate(results["documents"][0]):
        print(f"\nResult {i+1}:")
        print(document[:300])


QUESTION: What is machine learning?

Result 1:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
le

Result 2:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam,

Result 3:
not spam. The model learns statistical patterns that help classify new messages.
Traditional programming vs machine learning
Traditional programming
Machine learning
Rules + input → output
Examples + expected outputs → learned model
A developer specifies decision logic
An algorithm estimates useful 

QUESTION: Explain neural n

In [63]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"source": "machine_learning.pdf"}
)

In [64]:
%pip install -U groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Ijjat me learning\RAG Project\rag_project\Scripts\python.exe: No module named pip


In [81]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("GROQ client initialized")

GROQ client initialized


In [82]:
response=groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain machine learning in simple words."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

**Machine learning is a way to let computers learn from examples instead of being told exactly what to do.**  

Think of it like teaching a child to recognize a cat: you show many pictures of cats and tell the child, “This is a cat.” After seeing enough examples, the child can spot a cat in a new picture on their own.  

In machine learning, the computer is the child, the data (images, numbers, text, etc.) are the examples, and the “learning” is the process of finding patterns in that data. Once it has learned, the computer can make predictions or decisions—like recommending a movie, translating a sentence, or spotting a spam email—without being explicitly programmed for each specific case.


In [67]:
def retrieve_context(question, top_k=3):

    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_documents = []

    for i in range(len(results["documents"][0])):

        retrieved_documents.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i]
        })

    return retrieved_documents

In [68]:
question = "What is machine learning?"

context = retrieve_context(question)

for i, doc in enumerate(context):
    print(f"\nChunk {i+1}:")
    print(doc)


Chunk 1:
{'text': 'Learning notes \x7f Beginner to intermediate\nPage 2\n1. What is machine learning?\nMachine learning (ML) is a field of computer science in which a system learns patterns from data and uses those\npatterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a\nlearning algorithm.\nExample: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or', 'source': 'machine_learning.pdf', 'distance': 0.18527066707611084}

Chunk 2:
{'text': 'examples. Performance depends on data quality, assumptions and evaluation.\n2. Main types of machine learning\nSupervised learning\nThe training data contains input features and a target label/value. The model learns a mapping from inputs to\ntargets.\n\x7f\nClassification predicts a category: spam/not spam, churn/no churn, disease class.\n\x7f\nRegression predicts a numeric value: house price, delivery time, sales.\nUnsupervised learning', 'sou

In [69]:
def create_prompt(question, context):

    context_text = "\n\n".join(
        chunk["text"] for chunk in context
    )

    prompt = f"""
    You are an AI assistant that answers questions
    using the provided context.

    Rules:
    1. Answer only using the provided context.
    2. Do not make up information.
    3. If the answer is unavailable, say so.

    Context:
    {context_text}

    Question:
    {question}

    Answer:
    """

    return prompt

In [107]:
def generate_answer(question, context):

    prompt=create_prompt(question, context)

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [71]:
def rag_pipeline(question):
    # Retrieve relevant chunks
    context=retrieve_context(question)

    # Generate answer using retrieved context
    answer = generate_answer(question, context)

    return answer

In [72]:
question="How to play chess?"

answer=rag_pipeline(question)

print("Question: ", question)
print("Answer: ", answer)

Question:  How to play chess?
Answer:  I’m sorry, but I don’t have that information.


In [73]:
question = "What is Machine Learning"

results = retrieve_context(question, top_k=3)

for i, result in enumerate(results):
    print(f"\nChunk {i+1}:")
    print(result)


Chunk 1:
{'text': 'Learning notes \x7f Beginner to intermediate\nPage 2\n1. What is machine learning?\nMachine learning (ML) is a field of computer science in which a system learns patterns from data and uses those\npatterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a\nlearning algorithm.\nExample: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or', 'source': 'machine_learning.pdf', 'distance': 0.1964913010597229}

Chunk 2:
{'text': 'examples. Performance depends on data quality, assumptions and evaluation.\n2. Main types of machine learning\nSupervised learning\nThe training data contains input features and a target label/value. The model learns a mapping from inputs to\ntargets.\n\x7f\nClassification predicts a category: spam/not spam, churn/no churn, disease class.\n\x7f\nRegression predicts a numeric value: house price, delivery time, sales.\nUnsupervised learning', 'sour

In [74]:
print(results[0])
print(type(results))
print(type(results[0]))

{'text': 'Learning notes \x7f Beginner to intermediate\nPage 2\n1. What is machine learning?\nMachine learning (ML) is a field of computer science in which a system learns patterns from data and uses those\npatterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a\nlearning algorithm.\nExample: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or', 'source': 'machine_learning.pdf', 'distance': 0.1964913010597229}
<class 'list'>
<class 'dict'>


In [75]:
question = "What is machine learning?"

results = retrieve_context(question, top_k=5)

for i, result in enumerate(results):

    print(f"\nChunk {i+1}")
    print("Source:", result["source"])
    print("Distance:", round(result["distance"], 4))
    print("Text:", result["text"][:250])


Chunk 1
Source: machine_learning.pdf
Distance: 0.1853
Text: Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writin

Chunk 2
Source: machine_learning.pdf
Distance: 0.438
Text: examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.



Chunk 3
Source: machine_learning.pdf
Distance: 0.4381
Text: not spam. The model learns statistical patterns that help classify new messages.
Traditional programming vs machine learning
Traditional programming
Machine learning
Rules + input → output
Examples + expected outputs → learned model
A developer speci

Chunk 4
Source: machine_learning.pdf
Distance: 0.5366
Text: is r

In [76]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "C:/Ijjat me learning/RAG Project/documents/machine_learning.pdf"

reader = PdfReader(pdf_path)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

documents = []
metadatas = []
ids = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if not page_text:
        continue

    page_chunks = text_splitter.split_text(page_text)

    for chunk_number, chunk in enumerate(page_chunks):

        documents.append(chunk)

        metadatas.append({
            "source": Path(pdf_path).name,
            "page": page_number
        })

        ids.append(
            f"page_{page_number}_chunk_{chunk_number}"
        )

print("Total chunks:", len(documents))
print("Total metadata:", len(metadatas))

Total chunks: 27
Total metadata: 27


In [77]:
page_embeddings = model.encode(
    documents,
    show_progress_bar=True
)

print("Embedding shape:", page_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:04<00:00,  4.29s/it]

Embedding shape: (27, 384)


In [83]:
page_collection = chroma_client.get_or_create_collection(
    name="rag_with_page_metadata",
    metadata={"hnsw:space": "cosine"}
)

page_collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=page_embeddings.tolist(),
    metadatas=metadatas
)

print("Stored records:", page_collection.count())

Stored records: 27


In [84]:
def retrieve_context_with_pages(question, top_k=5):

    query_embedding = model.encode(question).tolist()

    results = page_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for i in range(len(results["documents"][0])):

        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"],
            "distance": results["distances"][0][i]
        })

    return retrieved

In [85]:
question = "What is machine learning?"

results = retrieve_context_with_pages(question, top_k=5)

for i, result in enumerate(results):

    print(f"\nChunk {i+1}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Distance:", round(result["distance"], 4))
    print("Text:", result["text"][:250])


Chunk 1
Source: machine_learning.pdf
Page: 2
Distance: 0.1853
Text: Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writin

Chunk 2
Source: machine_learning.pdf
Page: 2
Distance: 0.438
Text: examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.



Chunk 3
Source: machine_learning.pdf
Page: 2
Distance: 0.4381
Text: not spam. The model learns statistical patterns that help classify new messages.
Traditional programming vs machine learning
Traditional programming
Machine learning
Rules + input → output
Examples + expected outputs → learned model
A developer speci

Chunk 4
Source: machine_learning.pdf
Pag

In [87]:
def format_context(retrieved_chunks):

    context = ""

    for i, chunk in enumerate(retrieved_chunks):

        context += f"""
[Chunk {i+1}]
Source: {chunk['source']}
Page: {chunk['page']}

Content:
{chunk['text']}

-------------------------
"""

    return context

In [88]:
question = "What is machine learning?"

results = retrieve_context_with_pages(question, top_k=3)

context = format_context(results)

print(context)


[Chunk 1]
Source: machine_learning.pdf
Page: 2

Content:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
learning algorithm.
Example: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or

-------------------------

[Chunk 2]
Source: machine_learning.pdf
Page: 2

Content:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam, churn/no churn, disease class.

Regression predicts a numeric value: house price, delivery time, sales.
Unsupervised learn

In [90]:
def create_prompt(question, retrieved_chunks):

    context = format_context(retrieved_chunks)

    prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not make up information.
2. If the answer is not available in the context, say:
   "I don't have enough information to answer this question."
3. Include the source filename and page number for your answer.
4. Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [91]:
prompt = create_prompt(question, results)

print(prompt)


You are a helpful AI assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not make up information.
2. If the answer is not available in the context, say:
   "I don't have enough information to answer this question."
3. Include the source filename and page number for your answer.
4. Keep the answer clear and concise.

Context:

[Chunk 1]
Source: machine_learning.pdf
Page: 2

Content:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
learning algorithm.
Example: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or

-------------------------

[Chunk 2]
Source: machine_learning.pdf
Page: 2

Content:
examples. Performance depends on data quality, assumptions and evalua

In [92]:
def generate_answer(prompt):

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [93]:
def rag_pipeline(question, top_k=3):

    # 1. Retrieve relevant chunks
    retrieved_chunks = retrieve_context_with_pages(
        question,
        top_k=top_k
    )

    # 2. Create prompt
    prompt = create_prompt(
        question,
        retrieved_chunks
    )

    # 3. Generate answer
    answer = generate_answer(prompt)

    return answer

In [94]:
question = "What is machine learning?"

answer = rag_pipeline(question, top_k=3)

print("Question:", question)
print("\nAnswer:", answer)

Question: What is machine learning?

Answer: Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those patterns to make predictions or decisions, rather than having every rule written by hand.  
Source: *machine_learning.pdf*, page 2.


In [95]:
test_questions = [
    "What is machine learning?",
    "Explain supervised learning.",
    "What is the capital of Japan according to my documents?"
]

for question in test_questions:

    print("\n" + "="*60)
    print("QUESTION:", question)

    answer = rag_pipeline(question, top_k=3)

    print("\nANSWER:", answer)


QUESTION: What is machine learning?

ANSWER: Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a learning algorithm.  
**Source:** machine_learning.pdf, page 2

QUESTION: Explain supervised learning.

ANSWER: Supervised learning is a type of machine learning in which the training data contains input features and a target label or value. The model learns a mapping from the inputs to the targets, enabling it to make predictions on new data. It is commonly used for classification (e.g., spam/not‑spam) and regression (e.g., house price).  

Source: *machine_learning.pdf*, page 2.

QUESTION: What is the capital of Japan according to my documents?

ANSWER: I don't have enough information to answer this question.


In [96]:
question = "What is the capital of Japan?"

results = retrieve_context_with_pages(question, top_k=3)

for i, result in enumerate(results):

    print(f"\nChunk {i+1}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Distance:", round(result["distance"], 4))
    print("Text:", result["text"][:300])


Chunk 1
Source: machine_learning.pdf
Page: 3
Distance: 0.9246
Text: 
Split data into training, validation and test sets.

Build a simple baseline before trying complex models.

Train, tune and evaluate without leaking test information.

Analyse errors, document limitations and monitor performance after deployment.
Train, validation and test
Training data is used

Chunk 2
Source: machine_learning.pdf
Page: 3
Distance: 0.9326
Text: house-price dataset, area, location and number of rooms may be features; sale price is the target.
Common preprocessing

Missing values: impute with a justified statistic or use a model that handles them.

Categorical variables: one-hot encode nominal categories; use ordinal encoding only when ord

Chunk 3
Source: machine_learning.pdf
Page: 5
Distance: 0.9406
Text: y_train)\npred = model.predict(X_test)\nprint(classification_report(y_test, pred))
This example assumes all features are numeric. Real datasets often need separate preprocessing for categoric

In [97]:
def retrieve_context_with_pages(
    question,
    top_k=3,
    max_distance=0.7
):

    query_embedding = model.encode(question).tolist()

    results = page_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):

        distance = results["distances"][0][i]

        if distance <= max_distance:

            retrieved_chunks.append({
                "text": results["documents"][0][i],
                "source": results["metadatas"][0][i]["source"],
                "page": results["metadatas"][0][i]["page"],
                "distance": distance
            })

    return retrieved_chunks

In [98]:
def rag_pipeline(question, top_k=3):

    retrieved_chunks = retrieve_context_with_pages(
        question,
        top_k=top_k
    )

    if not retrieved_chunks:
        return "I don't have enough relevant information in my documents to answer this question."

    prompt = create_prompt(
        question,
        retrieved_chunks
    )

    answer = generate_answer(prompt)

    return answer

In [99]:
questions = [
    "What is machine learning?",
    "What is the capital of Japan?"
]

for question in questions:

    print("\nQuestion:", question)

    answer = rag_pipeline(question)

    print("Answer:", answer)


Question: What is machine learning?
Answer: Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a learning algorithm.  
Source: *machine_learning.pdf*, page 2

Question: What is the capital of Japan?
Answer: I don't have enough relevant information in my documents to answer this question.


In [100]:
def debug_retrieval(question, top_k=5, max_distance=0.7):

    query_embedding = model.encode(question).tolist()

    results = page_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    print("\nQUESTION:", question)
    print("\n" + "="*70)
    print("RETRIEVAL RESULTS")
    print("="*70)

    for i in range(len(results["documents"][0])):

        distance = results["distances"][0][i]
        text = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        is_relevant = distance <= max_distance

        print(f"\nChunk {i+1}")
        print("Source:", metadata["source"])
        print("Page:", metadata["page"])
        print("Distance:", round(distance, 4))
        print("Status:", "ACCEPTED" if is_relevant else "REJECTED")
        print("Text:", text[:300])

        if is_relevant:
            retrieved_chunks.append({
                "text": text,
                "source": metadata["source"],
                "page": metadata["page"],
                "distance": distance
            })

    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print("Total retrieved:", len(results["documents"][0]))
    print("Accepted:", len(retrieved_chunks))
    print("Rejected:", len(results["documents"][0]) - len(retrieved_chunks))

    return retrieved_chunks

In [101]:
question = "What is the capital of Japan?"

retrieved_chunks = debug_retrieval(
    question,
    top_k=5,
    max_distance=0.7
)


QUESTION: What is the capital of Japan?

RETRIEVAL RESULTS

Chunk 1
Source: machine_learning.pdf
Page: 3
Distance: 0.9246
Status: REJECTED
Text: 
Split data into training, validation and test sets.

Build a simple baseline before trying complex models.

Train, tune and evaluate without leaking test information.

Analyse errors, document limitations and monitor performance after deployment.
Train, validation and test
Training data is used

Chunk 2
Source: machine_learning.pdf
Page: 3
Distance: 0.9326
Status: REJECTED
Text: house-price dataset, area, location and number of rooms may be features; sale price is the target.
Common preprocessing

Missing values: impute with a justified statistic or use a model that handles them.

Categorical variables: one-hot encode nominal categories; use ordinal encoding only when ord

Chunk 3
Source: machine_learning.pdf
Page: 5
Distance: 0.9406
Status: REJECTED
Text: y_train)\npred = model.predict(X_test)\nprint(classification_report(y_test, pred

In [102]:
def rag_pipeline(question, top_k=3):

    # Step 1: Retrieve and filter chunks
    retrieved_chunks = debug_retrieval(
        question,
        top_k=top_k,
        max_distance=0.7
    )

    # Step 2: Stop if no relevant chunks
    if not retrieved_chunks:
        print("\nNo chunks sent to Groq!")
        return "I don't have enough relevant information in my documents."

    # Step 3: Format context
    context = format_context(retrieved_chunks)

    print("\n" + "="*70)
    print("CONTEXT ACTUALLY SENT TO GROQ")
    print("="*70)

    print(context)

    # Step 4: Create prompt
    prompt = create_prompt(question, retrieved_chunks)

    # Step 5: Generate answer
    print("\nSending prompt to Groq...")

    answer = generate_answer(prompt)

    return answer

In [104]:
answer = rag_pipeline("What is chess")

print("\nFINAL ANSWER:")
print(answer)


QUESTION: What is chess

RETRIEVAL RESULTS

Chunk 1
Source: machine_learning.pdf
Page: 2
Distance: 0.791
Status: REJECTED
Text: Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
le

Chunk 2
Source: machine_learning.pdf
Page: 2
Distance: 0.7955
Status: REJECTED
Text: Reinforcement learning
An agent takes actions in an environment, receives rewards or penalties, and learns a policy that aims to maximise
long-term reward. It is used in robotics, games and sequential decision-making.
Choose the learning setup
Problem
Likely setup
Example
Predict a numeric target
Su

Chunk 3
Source: machine_learning.pdf
Page: 2
Distance: 0.8192
Status: REJECTED
Text: not spam. The model learns statistical patterns that help classify new messages.
Traditional prog